In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import tensorflow as tf
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
import pickle


In [3]:
data=pd.read_csv('Churn_Modelling.csv')

In [4]:
data=data.drop(['RowNumber','CustomerId','Surname'], axis=1)
label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])
oh_encoder = OneHotEncoder()
geo_encoder = oh_encoder.fit_transform(data[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=oh_encoder.get_feature_names_out(['Geography']))
data = pd.concat([data.drop('Geography', axis=1) ,geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)

In [5]:
y= data['Exited']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y , test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
with open ('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
with open ('oh_encoder.pkl', 'wb') as f:
    pickle.dump(oh_encoder, f)
with open ('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [22]:
## define a function to create the model and try different parameters (kerasclassifier)
def create_model(neurons=32, layers=1):
    model=Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(neurons, activation='relu') )
    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))
    
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [23]:
model = KerasClassifier(layers=1, neurons=32,build_fn=create_model, verbose=0)

In [14]:
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1,2,3],
    'epochs': [50,100]
}

In [24]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X_train, y_train)

c:\Users\sarth\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


In [ ]:
print(f"best: {grid_result.best_score_} using {grid_result.best_params_}")

best: 0.8596244017884862 using {'epochs': 100, 'layers': 1, 'neurons': 64}


: 